# MLX System Model：数据 · 资源 · 执行 · 通信 · 优化

目标：建立 MLX 的**系统心智模型**——先把五层拆开，再看一次训练步如何穿过各层。

假设：本机同时可用 `mx.cpu` 与 `mx.gpu`（Apple Silicon / Metal）。无 GPU 环境请勿直接跑本笔记。

```text
                 MLX System Model

                   数据世界
                   Data World
                        |
                        v
                   资源世界
                   Resource World
                        |
                        v
                   执行世界
                   Execution World
                        |
                        v
                   通信世界
                   Communication World
                        |
                        v
                   优化世界
                   Optimization World
```

| 世界 | 回答的问题 | 本层对象 | 本层不负责 |
| --- | --- | --- | --- |
| **数据世界 Data** | 哪些样本由谁处理？ | Dataset、Shard | 设备、队列、归约、更新策略 |
| **资源世界 Resource** | 有哪些计算资源？算子落在哪？ | Device（cpu / gpu） | 数据怎么切、队列怎么排 |
| **执行世界 Execution** | 任务如何入队与跑完？ | Stream、kernel | 跨进程归约、Optimizer 策略 |
| **通信世界 Communication** | 多路如何同步/归约？ | `distributed` group、`all_sum` / average_gradients | 参数更新公式、数据切分本身 |
| **优化世界 Optimization** | 如何根据梯度更新参数？ | Optimizer、Parameter Update | Device 枚举、collective 语义 |

正交关系：

- `shard ≠ device`（Data ≠ Resource）——中间是 **runtime mapping**
- `device ≠ stream`（Resource ≠ Execution）
- `stream ≠ hardware`
- 通信（归约）≠ 优化（用归约后的梯度改参数）
- Optimizer **不属于** Execution 的一条 Stream 身份；它是 Optimization 层的步骤（其内部 update 仍会再变成 Execution 里的 kernels）

全链路接线见文末图 F。下文按五层分别跑通；通信做 `all_sum` 探活，优化给出编号步骤 + 单步 update。

依赖：`mlx`（含 `mlx.nn` / `mlx.optimizers`；§4 探活用 `mx.distributed`）。本笔记是**单进程内编排**，不替代 `mlx.launch` 多进程 DP。多进程分片编排见 [04](04_mlx_data_parallelism.ipynb) / [`dp_train_min.py`](dp_train_min.py)。

## 0. Setup

打印本机 Device 信息，并定义展示用小工具。

In [20]:
import mlx.core as mx
import mlx.nn as nn
import mlx.optimizers as optim


def report(name, got, expect):
    print(f"{name}:", got, "| expect:", expect)


def report_streams(dev, s1, s2, title="streams"):
    print(f"[{title}]")
    print("  Device:", dev)
    print("  s1:", s1, "device=", s1.device)
    print("  s2:", s2, "device=", s2.device)
    print("  same_device:", s1.device == s2.device)
    print("  same_stream:", s1 == s2)


print("Default device:")
print(mx.default_device())

print("\nAvailable devices:")
print("CPU count:", mx.device_count(mx.cpu))
print("GPU count:", mx.device_count(mx.gpu))

print("\nDevice objects:")
print("CPU:", mx.cpu)
print("GPU:", mx.gpu)

Default device:
Device(gpu, 0)

Available devices:
CPU count: 1
GPU count: 1

Device objects:
CPU: DeviceType.cpu
GPU: DeviceType.gpu


## 1. 数据世界（Data World）

回答：哪些样本由谁处理？

本节只做数据切分，**不出现** `stream=` / Device / `new_stream`。
与 [04](04_mlx_data_parallelism.ipynb) 的 `X[rank::size]` 同一切分语义；此处不 `distributed.init`。

In [21]:
X = mx.arange(12)
world_size = 3
shards = [X[r::world_size] for r in range(world_size)]

print("数据世界")
print(" Dataset", X)
for rank, shard in enumerate(shards):
    print(f"  +-- Shard{rank}", shard)

report("Shard0", shards[0], "[0, 3, 6, 9]")
report("Shard1", shards[1], "[1, 4, 7, 10]")
report("Shard2", shards[2], "[2, 5, 8, 11]")

数据世界
 Dataset array([0, 1, 2, ..., 9, 10, 11], dtype=int32)
  +-- Shard0 array([0, 3, 6, 9], dtype=int32)
  +-- Shard1 array([1, 4, 7, 10], dtype=int32)
  +-- Shard2 array([2, 5, 8, 11], dtype=int32)
Shard0: array([0, 3, 6, 9], dtype=int32) | expect: [0, 3, 6, 9]
Shard1: array([1, 4, 7, 10], dtype=int32) | expect: [1, 4, 7, 10]
Shard2: array([2, 5, 8, 11], dtype=int32) | expect: [2, 5, 8, 11]


## 2. 资源世界（Resource World）

回答：有哪些计算资源？这次计算落在哪？

- Array / Shard **不属于** GPU；`stream=Device` 指定的是**这次计算**落在哪块资源。
- Shard → Device 的连线叫 **runtime mapping**（编排选择，可改），不是数据世界的固有属性。
- 本节不讲多 stream，不做数据分片。

In [22]:
print("资源世界")
print(" +-- Device:", mx.cpu)
print(" +-- Device:", mx.gpu)

a = mx.ones((64, 64))
b = mx.ones((64, 64))
y_cpu = mx.matmul(a, b, stream=mx.cpu)
y_gpu = mx.matmul(a, b, stream=mx.gpu)
mx.eval(y_cpu, y_gpu)
report("cpu vs gpu matmul allclose", bool(mx.allclose(y_cpu, y_gpu)), True)

# 同一切片可换 Device 重算：数据对象不变，只改 runtime mapping
shard0 = mx.arange(6).astype(mx.float32)
r_cpu = mx.multiply(shard0, 2, stream=mx.cpu)
r_gpu = mx.multiply(shard0, 2, stream=mx.gpu)
mx.eval(r_cpu, r_gpu)
report("same shard, different device", bool(mx.allclose(r_cpu, r_gpu)), True)

资源世界
 +-- Device: DeviceType.cpu
 +-- Device: DeviceType.gpu
cpu vs gpu matmul allclose: True | expect: True
same shard, different device: True | expect: True


## 3. 执行世界（Execution World）

回答：同一个 Device 上，任务如何入队与跑完？

- **Device**：哪间厨房（计算资源，属资源世界）
- **Stream**：厨房里的哪条流水线（工作队列）
- 同 stream 内有序；不同 stream 可重叠调度
- 叶节点是 **kernel**（matmul / add / multiply），不是 Optimizer
- 本节用显式 `stream=...` 指定队列

In [23]:
# 同 Device，两条 Stream：可重叠入队
dev = mx.gpu
s1 = mx.new_stream(dev)
s2 = mx.new_stream(dev)
n = 512
a = mx.ones((n, n))
b = mx.ones((n, n))

report_streams(dev, s1, s2, title="执行世界：同 Device 两 Stream")

c = mx.matmul(a, b, stream=s1)
c2 = mx.add(c, 1, stream=s1)
d = mx.matmul(a, b, stream=s2)
d2 = mx.multiply(d, 2, stream=s2)
mx.eval(c2, d2)

print("执行世界")
print(f" Device = {dev}")
print("  +-- Stream0 --> matmul --> add")
print("  +-- Stream1 --> matmul --> multiply")

report("c2.mean (s1: matmul→add)", float(c2.mean()), n + 1)
report("d2.mean (s2: matmul→multiply)", float(d2.mean()), 2 * n)

[执行世界：同 Device 两 Stream]
  Device: DeviceType.gpu
  s1: Stream(Device(gpu, 0), 22) device= Device(gpu, 0)
  s2: Stream(Device(gpu, 0), 23) device= Device(gpu, 0)
  same_device: True
  same_stream: False
执行世界
 Device = DeviceType.gpu
  +-- Stream0 --> matmul --> add
  +-- Stream1 --> matmul --> multiply
c2.mean (s1: matmul→add): 513.0 | expect: 513
d2.mean (s2: matmul→multiply): 1024.0 | expect: 1024


In [24]:
# 同 Stream：有依赖则队列内串行
dev = mx.gpu
s1 = mx.new_stream(dev)
s2 = mx.new_stream(dev)
x = mx.array([1.0, 2.0, 3.0])

report_streams(dev, s1, s2, title="执行世界：同 Stream 串行")

y = mx.add(x, 1, stream=s1)
z = mx.multiply(y, 2, stream=s1)
mx.eval(z)
report("z", z, "[4, 6, 8]")

[执行世界：同 Stream 串行]
  Device: DeviceType.gpu
  s1: Stream(Device(gpu, 0), 24) device= Device(gpu, 0)
  s2: Stream(Device(gpu, 0), 25) device= Device(gpu, 0)
  same_device: True
  same_stream: False
z: array([4, 6, 8], dtype=float32) | expect: [4, 6, 8]


In [25]:
# 显式指定 Stream
dev = mx.gpu
s1 = mx.new_stream(dev)
base = mx.array([1.0, 2.0, 3.0])

r1 = mx.add(base, 1, stream=s1)
mx.eval(r1)
report("stream=s1", r1, "[2, 3, 4]")

stream=s1: array([2, 3, 4], dtype=float32) | expect: [2, 3, 4]


## 4. 通信世界（Communication World）

回答：多路如何同步 / 归约？

本层对象：`mx.distributed` group、`all_sum`（以及训练里常见的 `nn.average_gradients`）。

- **负责**：跨进程把张量汇合（如梯度 all-reduce）
- **不负责**：用汇合后的梯度怎么改参数（那是优化世界）

下方仅做归约 API 探活（Notebook 单进程时 `world.size()==1`，`all_sum` 近似 noop）。

In [26]:
world = mx.distributed.init()
print(f"rank={world.rank()} size={world.size()}")

probe = mx.distributed.all_sum(mx.ones(4))
mx.eval(probe)
print("all_sum(ones) =", probe)
print("(Notebook 单进程时为 1；mlx.launch -n 3 时为 3)")

rank=0 size=1
all_sum(ones) = array([1, 1, 1, 1], dtype=float32)
(Notebook 单进程时为 1；mlx.launch -n 3 时为 3)


## 5. 优化世界（Optimization World）

回答：如何根据梯度更新参数？

本层对象：**Optimizer**、**Parameter Update**。吃的是（通信世界归约后的）梯度；Optimizer **不是** Execution 里的第三条 Stream。

```text
优化一步（Optimization step）
  1. loss, grads = loss_and_grad(model, batch)   # 得到本地梯度
  2. # 若多进程：grads = average_gradients(grads)  # 属通信世界，本节省略
  3. optimizer.update(model, grads)              # 按策略写新参数
  4. mx.eval(model.parameters(), optimizer.state) # 物化（落到资源/执行）
```

In [27]:
# 最小单步：线性层 + MSE，只跑一步 update
class Tiny(nn.Module):
    def __init__(self):
        super().__init__()
        self.w = mx.zeros((2, 1))

    def __call__(self, x):
        return x @ self.w


def loss_fn(model, x, y):
    return mx.mean((model(x) - y) ** 2)


model = Tiny()
optimizer = optim.SGD(learning_rate=0.1)
loss_and_grad = nn.value_and_grad(model, loss_fn)

x = mx.array([[1.0, 0.0], [0.0, 1.0]])
y = mx.array([[2.0], [4.0]])
w_before = mx.array(model.w)

# 1) 求梯度
loss, grads = loss_and_grad(model, x, y)
# 2) 多进程时在此 average_gradients(grads) —— 属通信世界，本节省略
# 3) 按策略写新参数
optimizer.update(model, grads)
# 4) 物化
mx.eval(model.parameters(), optimizer.state)

print("step1 loss:", float(loss))
report("w before", w_before.squeeze(), "[0, 0]")
report("w after", model.w.squeeze(), "moved toward [2, 4]")

step1 loss: 10.0
w before: array([0, 0], dtype=float32) | expect: [0, 0]
w after: array([0.2, 0.4], dtype=float32) | expect: moved toward [2, 4]


## 6. 前三层编排 Demo（Runtime mapping）

串起 Data → Resource → Execution（图 F 上半段，到 kernel）。
Shard → Device 是 **runtime mapping**（可改），不是身份相等。

In [ ]:
# 1) 数据世界
X = mx.arange(12).astype(mx.float32)
shard0, shard1 = X[:6], X[6:]

# 2) 资源世界
device0 = mx.cpu
device1 = mx.gpu

# 3) 执行世界
s0 = mx.new_stream(device0)
s1 = mx.new_stream(device1)
out0 = mx.multiply(shard0, 2, stream=s0)
out1 = mx.multiply(shard1, 3, stream=s1)
mx.eval(out0, out1)

print("[Data]      shard0", shard0)
print("[Data]      shard1", shard1)
print("[Resource]  device0", device0, "| device1", device1)
print("[Execution] s0", s0, "| s1", s1)
print()
print("Dataset")
print("  +-- shard0 --runtime mapping-->", device0, "-->", s0, "--> multiply(*2)")
print("  +-- shard1 --runtime mapping-->", device1, "-->", s1, "--> multiply(*3)")
print()
report("out0", out0, "[0, 2, 4, 6, 8, 10]")
report("out1", out1, "[18, 21, 24, 27, 30, 33]")

# 正交性：同 shard 换 Device，结果不变
alt_dev = mx.gpu
out0_alt = mx.multiply(shard0, 2, stream=alt_dev)
mx.eval(out0_alt)
report("same shard0, remap device", bool(mx.allclose(out0, out0_alt)), True)

# 正交性：同 Device 新 Stream
s_extra = mx.new_stream(device0)
report("same device, new stream", s_extra != s0, True)

## 7. 全链路图 F + 收尾

垂直五层是**分类法**；下图是一次（数据并行）训练步如何穿过 System Model。
Shard → Device 标注 **runtime mapping**（虚线 / 可改）。

```mermaid
flowchart TB
    Dataset["Dataset"]
    Dataset --> S0["Shard0"]
    Dataset --> S1["Shard1"]
    S0 -.->|"runtime mapping"| D0["Device0"]
    S1 -.->|"runtime mapping"| D1["Device1"]
    D0 --> ST00["Stream0"]
    D0 --> ST01["Stream1"]
    D1 --> ST10["Stream0"]
    D1 --> ST11["Stream1"]
    ST00 --> K00["kernel"]
    ST01 --> K01["kernel"]
    ST10 --> K10["kernel"]
    ST11 --> K11["kernel"]
    K00 --> Comm["Communication all_sum"]
    K01 --> Comm
    K10 --> Comm
    K11 --> Comm
    Comm --> Opt["Optimizer"]
    Opt --> PU["Parameter Update"]
```

分层读图：

- Dataset / Shard → **数据世界**
- Device → **资源世界**
- Stream / kernel → **执行世界**
- `all_sum` / average_gradients → **通信世界**
- Optimizer / Parameter Update → **优化世界**（步骤见 §5）

### 收尾要点

- §6 已在单进程内跑通 Data → Resource → Execution 编排；§4 通信探活、§5 优化单步均在本笔记内可跑。
- 多进程分片编排见 [04](04_mlx_data_parallelism.ipynb) / [`dp_train_min.py`](dp_train_min.py)（`mlx.launch -n 3`）。
- 本笔记不替代多进程 DP；`mapping` 可改，五层对象正交。
- Lazy：构图多半是入队；真正算完靠 `mx.eval` / `mx.synchronize`。